<a href="https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2004%20-%20A%20Matrix%20Can%20Transform%20Space/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lecture 04 — A Matrix Can Transform Space · Laboratory

↩ **Theory:** [`blog.md`](<blog.md>) — read the lecture first. This notebook tests it.

## Step 1 — The Problem

Chapter 3 said a matrix stores a table. It does something far more interesting: it picks up
**every point in space at once** and puts it somewhere else.

$$A = \begin{bmatrix} 2 & 0 \\ 0 & 3\end{bmatrix}, \qquad A\begin{bmatrix}1\\1\end{bmatrix} = \begin{bmatrix}2\\3\end{bmatrix}$$

Three questions: what kinds of movement are possible, does doing $A$ then $B$ equal one
matrix, and **does chaining many matrices build anything new?**

## Step 2 — Prediction

Commit before running. You check these in Step 10.

1. A matrix has 4 numbers but moves infinitely many points. What 2 things do you need to
   know to predict where *every* point goes?
2. The shear $\begin{bmatrix}1&1\\0&1\end{bmatrix}$ slants the square. Does it change its **area**?
3. Is rotating-then-shearing the same as shearing-then-rotating?
4. Three matrices applied in a row: is the result more powerful than one matrix, or not?

In [ ]:
# Step 3 — Intuition: the columns are where the basis vectors land.
import numpy as np
np.random.seed(0)

A = np.array([[2., 0.], [0., 3.]])
e1, e2 = np.array([1., 0.]), np.array([0., 1.])

print("A @ e1 =", A @ e1, "  <- column 1 of A:", A[:, 0])
print("A @ e2 =", A @ e2, "  <- column 2 of A:", A[:, 1])
assert np.array_equal(A @ e1, A[:, 0])
assert np.array_equal(A @ e2, A[:, 1])

# And every other point follows: x = x1*e1 + x2*e2, so Ax = x1*col1 + x2*col2
x = np.array([3., 7.])
assert np.allclose(A @ x, x[0] * A[:, 0] + x[1] * A[:, 1])
print("\nA @ [3,7] =", A @ x, "= 3*col1 + 7*col2 =", 3 * A[:, 0] + 7 * A[:, 1])

## Step 4 — The Mathematics Under Test

$$A\mathbf{e}_j = \mathbf{a}_j \quad(\text{column } j)
\qquad
\det\begin{bmatrix}a&b\\c&d\end{bmatrix} = ad-bc
\qquad
A(B\mathbf{x}) = (AB)\mathbf{x}$$

Step 5 checks every number the lecture claims.

In [ ]:
# Step 5 — The four transformations from blog section 6, read off their columns.
scale  = np.array([[2., 0.], [0., 3.]])
rotate = np.array([[0., -1.], [1., 0.]])     # 90 degrees anticlockwise
shear  = np.array([[1., 1.], [0., 1.]])
project = np.array([[1., 0.], [0., 0.]])     # flatten onto the x-axis

for name, M in [("scale", scale), ("rotate", rotate), ("shear", shear), ("project", project)]:
    print(f"{name:>8}:  e1 -> {M @ e1},  e2 -> {M @ e2},  det = {np.linalg.det(M):+.1f}")

# The determinant is the area-scaling factor (blog section 7).
assert np.isclose(np.linalg.det(scale), 6.0)      # 2 x 3 rectangle
assert np.isclose(np.linalg.det(rotate), 1.0)     # turning changes no area
assert np.isclose(np.linalg.det(shear), 1.0)      # same base, same height
assert np.isclose(np.linalg.det(project), 0.0)    # flattened to a line

In [ ]:
# Step 5b — when a matrix destroys information (blog section 8).
# The projection sends an entire vertical line onto one point.
for y in [0., 9., 100., -3.]:
    print(f"[5, {y:>6}] -> {project @ np.array([5., y])}")

print("\nrank(project) =", np.linalg.matrix_rank(project), " (only 1 direction survives)")
print("rank(shear)   =", np.linalg.matrix_rank(shear))
assert np.linalg.matrix_rank(project) == 1
assert np.linalg.matrix_rank(shear) == 2

# rank + nullity = number of input dimensions
nullity = 2 - np.linalg.matrix_rank(project)
assert np.linalg.matrix_rank(project) + nullity == 2
print("rank + nullity =", np.linalg.matrix_rank(project), "+", nullity, "= 2")

# det = 0 means no inverse exists: the information is gone, not hidden.
try:
    np.linalg.inv(project)
    raise SystemExit("should not get here")
except np.linalg.LinAlgError:
    print("\nproject has no inverse, as det = 0 predicts")

In [ ]:
# Step 5c — composition IS matrix multiplication (blog section 9), and order matters.
x = np.array([2., 1.])
assert np.allclose(shear @ (rotate @ x), (shear @ rotate) @ x)
print("shear(rotate(x)) == (shear @ rotate) x  ->", shear @ (rotate @ x))

SR = shear @ rotate
RS = rotate @ shear
print("\nSR =\n", SR, "\nRS =\n", RS)
assert np.array_equal(SR, [[1., -1.], [1., 0.]])
assert np.array_equal(RS, [[0., -1.], [1., 1.]])
assert not np.array_equal(SR, RS)
print("\nSocks then shoes is not shoes then socks.")

In [ ]:
# Step 7 — Visualization: watch the unit square move.
import matplotlib.pyplot as plt

square = np.array([[0., 0.], [1., 0.], [1., 1.], [0., 1.], [0., 0.]]).T  # 2 x 5

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
panels = [("original", np.eye(2)), ("scale", scale), ("rotate", rotate),
          ("shear", shear), ("project", project)]
for ax, (name, M) in zip(axes, panels):
    out = M @ square
    ax.plot(square[0], square[1], "--", color="grey", lw=1)
    ax.fill(out[0], out[1], alpha=.3)
    ax.plot(out[0], out[1], lw=2)
    ax.arrow(0, 0, *(M @ e1), head_width=.12, color="crimson", length_includes_head=True)
    ax.arrow(0, 0, *(M @ e2), head_width=.12, color="darkgreen", length_includes_head=True)
    ax.set_title(f"{name}  (det={np.linalg.det(M):+.0f})", fontsize=10)
    ax.set_xlim(-2.2, 3.2); ax.set_ylim(-1.6, 3.4)
    ax.set_aspect("equal"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# Red = where e1 landed, green = where e2 landed. Those two arrows ARE the matrix.

In [ ]:
# Step 8 — The experiment: does depth buy anything? (blog section 11)
W1 = np.array([[1., 2.], [0., 1.]])
W2 = np.array([[2., 0.], [1., 1.]])
W3 = np.array([[0., 1.], [-1., 0.]])

x = np.array([3., 2.])
step_by_step = W3 @ (W2 @ (W1 @ x))
C = W3 @ W2 @ W1
one_shot = C @ x

print("three layers, one at a time:", step_by_step)
print("a single matrix C:         ", one_shot)
print("\nC =\n", C)
assert np.allclose(step_by_step, one_shot)
assert np.array_equal(C, [[1., 3.], [-2., -4.]])
print("\nIdentical. Three layers collapsed into four numbers.")

In [ ]:
# Step 9 — Change one variable: the number of stacked layers.
rng = np.random.default_rng(0)
print(f"{'layers':>8} {'composite is still 2x2?':>26} {'max error vs step-by-step':>28}")
for n in [2, 5, 20, 100]:
    mats = [rng.normal(size=(2, 2)) for _ in range(n)]
    v = rng.normal(size=2)

    out = v.copy()
    for M in mats:          # apply them one at a time
        out = M @ out

    C = np.eye(2)
    for M in mats:          # or multiply them all together first
        C = M @ C

    err = np.max(np.abs(out - C @ v))
    print(f"{n:>8} {str(C.shape):>26} {err:>28.2e}")
print("\nNo matter how deep the stack, the composite is always a single 2x2 matrix.")

## Step 10 — Observe

Against your Step 2 predictions:

1. You only need **two** things: where $\mathbf{e}_1$ lands and where $\mathbf{e}_2$ lands. Those
   are the columns, and Step 3 confirmed every other point follows from them.
2. The shear has $\det = 1$, so the area is **unchanged** — the square becomes a leaning
   parallelogram with the same base and height.
3. $SR \ne RS$. Two different matrices, so two different transformations.
4. **No.** A hundred stacked matrices collapse to one $2\times2$ matrix, to machine precision.

## Step 11 — Explain

**Why four numbers are enough.** Every point is $x_1\mathbf{e}_1 + x_2\mathbf{e}_2$, and a linear
map preserves that combination. So once you know where the two basis arrows go, the fate of
every other point is already decided. The columns are those two arrows.

**Why the determinant matters.** It is the factor by which area is multiplied. When it hits
zero, space has been squashed flat — an entire line of inputs maps to a single output, so no
inverse can exist. That is not lost precision; the information is genuinely gone.

**Why depth buys nothing.** Associativity lets you regroup
$W_3(W_2(W_1\mathbf{x}))$ into $(W_3W_2W_1)\mathbf{x}$, and the product of matrices is a matrix.

> A hundred linear layers with a million parameters can only ever do what four numbers can do.
> The depth is fictitious.

This is exactly why something non-linear must sit between the layers — and why Chapters 24
and 25 exist. Depth only becomes real once we break linearity.

In [ ]:
# Step 12 — Challenges.

# LEVEL 3 (Derive, then verify): prove det(AB) = det(A)det(B) for 2x2 matrices.
for _ in range(3):
    P, Q = rng.normal(size=(2, 2)), rng.normal(size=(2, 2))
    assert np.isclose(np.linalg.det(P @ Q), np.linalg.det(P) * np.linalg.det(Q))
print("det(AB) = det(A)det(B) holds")

# LEVEL 4 (Investigate): track the determinant of a growing chain.
# What happens to det(C) as the number of layers grows, and what does that
# predict about very deep linear networks?

# YOUR CODE HERE


# LEVEL 5 (Design): no 2x2 matrix can shift the square one step right, because
# every linear map fixes the origin. Prove it, then build a 3x3 matrix that
# does it for points written as (x, y, 1).
def translate(dx, dy):
    # YOUR CODE HERE
    ...

# Where have you already met this trick? Look at the +b in our model.

## Step 13 — Reflection

- [ ] I can read a matrix by asking where $\mathbf{e}_1$ and $\mathbf{e}_2$ land.
- [ ] I can say what $\det = 0$ means without using the word "determinant".
- [ ] I can explain why $AB \neq BA$ using an everyday example.
- [ ] I watched a hundred layers collapse into four numbers, and I can explain why.
- [ ] I know what must be added between layers to make depth real.

### The question this chapter leaves open

The shear left the $x$-axis exactly where it was. The scaling matrix left both axes pointing
the same way, changing only their lengths. The rotation left **nothing** unmoved.

So some directions are special to a particular matrix — it stretches them without turning
them. Find those, and a complicated transformation becomes "stretch by this much along
these few directions."

➡️ **Next:** [Chapter 05 — Eigenvectors: What a Transformation Leaves Alone](<../Lecture 05 - Eigenvectors: What a Transformation Leaves Alone/blog.md>)